# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [ ]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    # TODO: Handle empty input
    # TODO: Support: greet <name>
    # TODO: Support: add <x> <y>
    # TODO: Use try/except for numeric conversion
    # TODO: Return an error/usage message for bad commands
    commands = ["greet", "add"]

    if not argv:
        return "Error: You have to add a value"

    if argv[0] not in commands:
        return "Error: Command not valid (type: greet or add)"
    
    if argv[0].lower() == commands[0]:
        return f"Hello, {argv[1]}!"
    
    elif argv[0].lower() == commands[1]:
        try:
            return float(argv[1]) + float(argv[2])
        except ValueError:
            return "Error: Input not valid for addition"

# --- TEST ---
print(run_command([]))

print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input

print(run_command(["subtract", "4", "3"]))
# Expected: Error message about numeric input


Error: You have to add a value
Hello, Alice!
7.0
Error: Input not valid for addition
Error: Command not valid (type: greet or add)


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [5]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # TODO: Create ArgumentParser
    # TODO: Add optional --category argument
    # TODO: Add optional --min argument (type=float, default=0)
    parser = argparse.ArgumentParser(description="Expense Categorizer")
    parser.add_argument("--category", help="Filter by category")
    parser.add_argument("--min", type=float, default=0, help="Minimum amount")

    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    # TODO: Loop through expenses and keep only matching rows
    result = []
    for e in expenses:
        if category is not None and e["category"].lower() != category.lower():
            continue 
        if e["amount"] < min_amount:
            continue
        result.append(e)

    return result
    

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # TODO: Handle empty list
    # TODO: Build a list of output lines
    # TODO: Return one multi-line string
    if not expenses:
        return "Empty expenses list"
    lines = [f"{'Date':<12}{'Category':<12}{'Amount':>8} Note", "-" * 20]
    for e in expenses:
        lines.append(f"{e['date']:<12}{e['category']:<12}{e['amount']:>8.2f} {e['note']}")
    lines.append("-"*20)
    lines.append(f"{'Total':<24}{sum(e['amount'] for e in expenses):>8.2f}")
    return "\n".join(lines)

# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


Date        Category      Amount Note
--------------------
2026-10-02  Food           34.20 Groceries
--------------------
Total                      34.20


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [8]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [10]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    # TODO: Use Path(filename).stem and .suffix
    # TODO: Add a timestamp with datetime.now().strftime(...)
    p = Path(filename)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{p.stem}_{stamp}{p.suffix}"

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    # TODO: Convert to Path objects
    # TODO: Check whether source exists
    # TODO: Create backup folder if needed
    # TODO: Copy file with shutil.copy2
    # TODO: Return destination path as a string
    source = Path(source_path)
    if not source.is_file():
        return None

    folder = Path(backup_dir)
    folder.mkdir(parents=True, exist_ok=True)

    destination = folder / make_backup_name(source.name)
    shutil.copy2(source, destination)
    return str(destination)

# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)


Backup created: backups/budget_data_20261110_075223.json


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [11]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [13]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    # TODO: Use csv.DictReader
    # TODO: Validate category and amount
    # TODO: Convert amount to float
    # TODO: Skip bad rows safely
    valid = []
    with open(filename, newline="", encoding="utf-8") as f:
        for line_no, row in enumerate(csv.DictReader(f), start=2):
            category = (row.get("category") or "").strip()
            if not category:
                print(f"Skipping line {line_no}: No category")
                continue
            try:
                amount = float(row.get("amount") or "")
            except ValueError:
                print(f"Skipping line {line_no}: Amount not float")
                continue
            valid.append({**row, "category": category, "amount": amount})

    return valid

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    # TODO: Accumulate totals in a dictionary
    totals = {}
    for e in expenses:
        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]
    return totals

def print_summary(summary):
    """Print a formatted category summary."""
    # TODO: Handle empty summary
    # TODO: Print aligned output
    if not summary:
        print("No expenses")
        return
    print(f"{'Category':<12}{'Total':>10}")
    print("-"*22)
    for cat, total in sorted(summary.items(), key=lambda kv: kv[1], reverse=True):
        print(f"{cat:<12}{total:>10.2f}")
    print("-"*22)
    print(f"{'Total':<12}{sum(summary.values()):>10.2f}")

# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


Skipping line 4: No category
Skipping line 5: Amount not float
Category         Total
----------------------
Food             39.80
Transport         2.80
----------------------
Total            42.60


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [14]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # TODO: Validate minute (0-59) and hour (0-23)
    # TODO: Return the cron line as a string
    if not isinstance(minute, int) or not 0 <= minute <= 59:
        raise ValueError(f"minute must be 0-50, minutes: {minute!r}")
    if not isinstance(hour, int) or not 0 <= hour <= 23:
        raise ValueError(f"hour must be 0-23, hours: {hour!r}")
    return f"{minute} {hour} * * * python {script_path}"

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # TODO: Validate HH:MM format
    # TODO: Return a command like:
    # schtasks /Create /SC DAILY /TN BudgetBackup /TR "python script.py" /ST 18:30
    if not isinstance(time_str, str) or len(time_str) != 5:
        raise ValueError(f"time must be HH:MM, got {time_str!r}")
    try:
        datetime.strptime(time_str, "%H:%M")
    except ValueError:
        raise ValueError(f"time must be a valid 24h HH:MM, got {time_str!r}")
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {script_path}" /ST {time_str}'

# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))


30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [15]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [16]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    # TODO: Handle FileNotFoundError and json.JSONDecodeError
    # TODO: Ensure the top-level object is a list
    # TODO: Return [] on error
    try:
        with open(filename, encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: {filename} not found")
        return []
    except json.JSONDecodeError as e:
        print(f"Error: {filename} is not valid JSON ({e})")
        return []
    if not isinstance(data, list):
        print("Error: expected a list of expenses")
        return []
    return data

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    # TODO: Create parser
    # TODO: Add a required filename argument
    # TODO: Add subcommands: list, summary, backup
    parser = argparse.ArgumentParser(prog="expenses", description="Expense automation suite")
    parser.add_argument("filename", help="Path to the JSON expense file")
    sub = parser.add_subparsers(dest="command", required=True)
    sub.add_parser("list", help="Print all expenses")
    sub.add_parser("summary", help="Show totals by category")
    sub.add_parser("backup", help="Make a timestamped backup")
    return parser

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    # TODO: Parse args
    # TODO: Load data when needed
    # TODO: Route to list / summary / backup
    parser = build_suite_parser()
    args = parser.parse_args(argv)

    if args.command == "backup":
        dest = backup_file(args.filename)
        return f"Backup created: {dest}" if dest else f"Error: {args.filename} not found"

    expenses = load_json_expenses(args.filename)

    if args.command == "list":
        return format_expenses(expenses)

    if args.command == "summary":
        summary = summarize_by_category(expenses)
        if not summary:
            return "No expenses to summarize."
        lines = [f"{'Category':<12}{'Total':>10}", "-" * 22]
        for cat, total in sorted(summary.items(), key=lambda kv: kv[1], reverse=True):
            lines.append(f"{cat:<12}{total:>10.2f}")
        lines.append("-" * 22)
        lines.append(f"{'TOTAL':<12}{sum(summary.values()):>10.2f}")
        return "\n".join(lines)


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))


Date        Category      Amount Note
--------------------
2026-10-01  Food           18.50 Lunch
2026-10-02  Transport       2.80 Metro
2026-10-03  Food           21.30 Dinner
2026-10-04  Books          45.00 Python book
--------------------
Total                      87.60

Category         Total
----------------------
Books            45.00
Food             39.80
Transport         2.80
----------------------
TOTAL            87.60

Backup created: backups/expenses_suite_20261110_080406.json
